## Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import os

print(os.getcwd())

D:\BERT_Project


In [3]:
data = pd.read_csv(
    r"D:\BERT_Project\Original Dataset\dataset\CivicDataset.csv"
)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   original_id              16000 non-null  object
 1   Complain_date            16000 non-null  object
 2   complaint_title          16000 non-null  object
 3   text_en                  16000 non-null  object
 4   location_address         8836 non-null   object
 5   category_primary         15561 non-null  object
 6   category_secondary       16000 non-null  object
 7   resolution_status        16000 non-null  object
 8   comment_count            16000 non-null  int64 
 9   Hindi                    16000 non-null  object
 10  Hinglish                 16000 non-null  object
 11  severity                 16000 non-null  object
 12  severity_score           16000 non-null  int64 
 13  high_keywords_matched    2816 non-null   object
 14  medium_keywords_matched  5328 non-null

In [5]:
data.describe()

,comment_count,severity_score
count,16000.000000,16000.000000
mean,1.254125,2.339500
std,2.784090,2.971875
min,0.000000,0.000000
25%,0.000000,0.000000
50%,1.000000,2.000000
75%,2.000000,4.000000
max,115.000000,23.000000


In [6]:
print("Number of Primary Categories:", data['category_primary'].nunique())
print("Primary Categories:")
print(data['category_primary'].unique())

print(data['category_primary'].value_counts())

Number of Primary Categories: 8
Primary Categories:
['Encroachment & Illegal Construction' 'Others' 'Road & Transportation'
 'Water Leakage' 'Waste Management' 'Electricity' 'Health & Sanitation'
 'Public Safety' nan]
category_primary
Road & Transportation                  6171
Waste Management                       4900
Electricity                            1741
Others                                 1457
Health & Sanitation                     550
Water Leakage                           341
Encroachment & Illegal Construction     237
Public Safety                           164
Name: count, dtype: int64


In [7]:
print("Number of Secondary Categories:", data['category_secondary'].nunique())
print("Primary Categories:")

print(data['category_secondary'].value_counts())

Number of Secondary Categories: 30
Primary Categories:
category_secondary
Others                                   6703
Garbage Dumping & Black Spots            3120
Road Damage & Potholes                   2252
Streetlights & Public Lighting           1452
Public Nuisance                           371
Garbage Collection                        365
Water Supply Issues                       242
Traffic Management & Violations           223
Footpaths & Pedestrian Infrastructure     211
Parking Issues                            210
Power Supply Issues                       192
Traffic Rule Violations                   158
Drainage & Sewage                          64
Public Toilets                             57
Water Leakage & Wastage                    56
Mosquito & Pest Control                    52
Land & Property Encroachment               49
Unauthorized Construction                  39
Water Quality & Metering                   30
Water Pipeline & Infrastructure            29
Waste 

In [8]:
# Count of each severity
print("Severity Counts:")
print(data['severity'].value_counts())

Severity Counts:
severity
LOW       10672
MEDIUM     3285
HIGH       2043
Name: count, dtype: int64


In [9]:
severity_summary = (
    data.groupby('severity')
        .agg(
            Count=('severity', 'size'),
            Avg_Score=('severity_score', 'mean'),
            Min_Score=('severity_score', 'min'),
            Max_Score=('severity_score', 'max')
        )
        .round(2)
)

print(severity_summary)

          Count  Avg_Score  Min_Score  Max_Score
severity                                        
HIGH       2043       8.57          6         23
LOW       10672       0.69          0          2
MEDIUM     3285       3.81          3          5


In [14]:
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split

# ============================================================
# Load Dataset
# ============================================================

data = pd.read_csv("Original Dataset/dataset/CivicDataset.csv")

print("Original Dataset Shape:", data.shape)

# ============================================================
# Create Primary -> Secondary Mapping
# ============================================================

primary_to_secondary = (
    data.groupby("category_primary")["category_secondary"]
        .apply(lambda x: sorted(x.dropna().unique()))
        .to_dict()
)

print("\n========== Primary → Secondary Mapping ==========\n")

for primary, secondary in primary_to_secondary.items():
    print(primary)
    for s in secondary:
        print(f"   - {s}")

# Save mapping for later synthetic generation

with open("primary_to_secondary.json", "w", encoding="utf-8") as f:
    json.dump(primary_to_secondary, f, indent=4, ensure_ascii=False)

print("\nMapping saved as primary_to_secondary.json")

# ============================================================
# Remove rows with missing target labels
# ============================================================

data = data.dropna(
    subset=[
        "category_primary",
        "category_secondary",
        "severity"
    ]
).reset_index(drop=True)

print("\nAfter Removing Missing Labels:", data.shape)

# ============================================================
# Create Combined Stratification Label
# ============================================================

data["stratify_label"] = (
    data["category_primary"].astype(str)
    + "__"
    + data["category_secondary"].astype(str)
    + "__"
    + data["severity"].astype(str)
)

# ============================================================
# Remove combinations occurring only once
# ============================================================

counts = data["stratify_label"].value_counts()

valid_labels = counts[counts >= 2].index

rare_samples = data[~data["stratify_label"].isin(valid_labels)]

data = data[
    data["stratify_label"].isin(valid_labels)
].reset_index(drop=True)

print("\nRemoved Rare Samples:", len(rare_samples))
print("Remaining Samples:", len(data))

# ============================================================
# Train (80%) + Temp (20%)
# ============================================================

train_df, temp_df = train_test_split(
    data,
    test_size=0.20,
    random_state=42,
    stratify=data["stratify_label"]
)

# ============================================================
# Dev (10%) + Test (10%)
# ============================================================

dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

# ============================================================
# Remove Helper Column
# ============================================================

for df in [train_df, dev_df, test_df]:
    df.drop(columns=["stratify_label"], inplace=True)

# ============================================================
# Create Folder Structure
# ============================================================

base_dir = "dataset"

os.makedirs(os.path.join(base_dir, "train"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "dev"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "test"), exist_ok=True)

# ============================================================
# Save CSV Files
# ============================================================

train_df.to_csv(
    os.path.join(base_dir, "train", "train.csv"),
    index=False
)

dev_df.to_csv(
    os.path.join(base_dir, "dev", "dev.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(base_dir, "test", "test.csv"),
    index=False
)

# ============================================================
# Save JSON Files
# ============================================================

train_df.to_json(
    os.path.join(base_dir, "train", "train.json"),
    orient="records",
    indent=4,
    force_ascii=False
)

dev_df.to_json(
    os.path.join(base_dir, "dev", "dev.json"),
    orient="records",
    indent=4,
    force_ascii=False
)

test_df.to_json(
    os.path.join(base_dir, "test", "test.json"),
    orient="records",
    indent=4,
    force_ascii=False
)

# ============================================================
# Summary
# ============================================================

print("\n================ Dataset Summary ================\n")

print("Train :", train_df.shape)
print("Dev   :", dev_df.shape)
print("Test  :", test_df.shape)

print("\nFiles Saved Successfully!")

print("\ndataset/")
print("├── train/")
print("│   ├── train.csv")
print("│   └── train.json")
print("├── dev/")
print("│   ├── dev.csv")
print("│   └── dev.json")
print("├── test/")
print("│   ├── test.csv")
print("│   └── test.json")
print("└── primary_to_secondary.json")

# ============================================================
# Verify Distribution
# ============================================================

def show_distribution(column):

    print("\n" + "=" * 70)
    print(column)
    print("=" * 70)

    dist = pd.DataFrame({
        "Train (%)": train_df[column].value_counts(normalize=True).mul(100).round(2),
        "Dev (%)": dev_df[column].value_counts(normalize=True).mul(100).round(2),
        "Test (%)": test_df[column].value_counts(normalize=True).mul(100).round(2),
    }).fillna(0)

    print(dist)

show_distribution("category_primary")
show_distribution("category_secondary")
show_distribution("severity")

Original Dataset Shape: (16000, 15)

========== Primary → Secondary Mapping ==========

Electricity
   - Electrical Infrastructure
   - Others
   - Power Supply Issues
   - Streetlights & Public Lighting
Encroachment & Illegal Construction
   - Land & Property Encroachment
   - Others
   - Unauthorized Construction
Health & Sanitation
   - Drainage & Sewage
   - Others
   - Public Hygiene
   - Public Toilets
Others
   - Animal & Organic Waste
   - Lake & Public Space Encroachment
   - Others
   - Public Transport Services
   - Road Damage & Potholes
Public Safety
   - Fire Safety
   - Others
   - Public Nuisance
Road & Transportation
   - Footpaths & Pedestrian Infrastructure
   - Others
   - Parking Issues
   - Road Damage & Potholes
   - Surveillance & Policing
   - Traffic Management & Violations
   - Traffic Rule Violations
Waste Management
   - Construction & Debris Waste
   - Drainage & Sewage
   - Garbage Collection
   - Garbage Dumping & Black Spots
   - Mosquito & Pest Control

In [15]:
### Check Hiererachy
primary_to_secondary = (
    data.groupby('category_primary')['category_secondary']
    .unique()
)

for primary, secondary in primary_to_secondary.items():
    print(f"\n{primary}")
    for s in sorted(secondary):
        print(f"   - {s}")


Electricity
   - Electrical Infrastructure
   - Others
   - Power Supply Issues
   - Streetlights & Public Lighting

Encroachment & Illegal Construction
   - Land & Property Encroachment
   - Others
   - Unauthorized Construction

Health & Sanitation
   - Drainage & Sewage
   - Others
   - Public Toilets

Others
   - Animal & Organic Waste
   - Lake & Public Space Encroachment
   - Others
   - Public Transport Services
   - Road Damage & Potholes

Public Safety
   - Fire Safety
   - Others
   - Public Nuisance

Road & Transportation
   - Footpaths & Pedestrian Infrastructure
   - Others
   - Parking Issues
   - Road Damage & Potholes
   - Surveillance & Policing
   - Traffic Management & Violations
   - Traffic Rule Violations

Waste Management
   - Drainage & Sewage
   - Garbage Collection
   - Garbage Dumping & Black Spots
   - Mosquito & Pest Control
   - Others
   - Street Cleanliness
   - Waste Segregation & Processing

Water Leakage
   - Others
   - Water Leakage & Wastage
   - 